# 01 — Exploratory Data Analysis

Load `power.pk`, inspect structure, handle missing values, visualise.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.loader import load_raw

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

## 1. Load raw data

In [ ]:
df = load_raw('../data/power.pk')
print('Shape (T x N_meters):', df.shape)
print('dtype :', df.dtypes.unique())
print('Index type:', type(df.index))
df.head()

In [ ]:
# Basic stats
T, N = df.shape
print(f'Meters: {N}')
print(f'Timesteps: {T}')
if hasattr(df.index, 'freq'):
    print(f'Freq: {df.index.freq}')
print(f'Date range: {df.index[0]}  →  {df.index[-1]}')
print(f'\nGlobal NaN fraction: {df.isna().values.mean():.3%}')
print(f'Meters with >10% NaN: {(df.isna().mean() > 0.1).sum()}')

## 2. Missing value heatmap

In [ ]:
# Resample to daily NaN fraction for a compact heatmap
if isinstance(df.index, pd.DatetimeIndex):
    daily_nan = df.isna().resample('D').mean()  # (days, meters)
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(daily_nan.T.values, aspect='auto', vmin=0, vmax=0.5, cmap='Reds')
    plt.colorbar(im, ax=ax, label='Daily NaN fraction')
    ax.set_xlabel('Day')
    ax.set_ylabel('Meter index')
    ax.set_title('Missing values per meter per day')
    plt.tight_layout()
    plt.show()
else:
    print('No DatetimeIndex — skip heatmap')

## 3. Sample time series plots

In [ ]:
rng = np.random.default_rng(42)
sample_meters = rng.choice(N, size=min(5, N), replace=False)

# Show 2 weeks
show_steps = min(T, 96 * 14)
fig, axes = plt.subplots(len(sample_meters), 1, figsize=(14, 2.5 * len(sample_meters)), sharex=True)
for ax, m in zip(axes, sample_meters):
    ax.plot(df.iloc[:show_steps, m].values, linewidth=0.7)
    ax.set_ylabel(f'Meter {m}', fontsize=9)
    ax.set_ylim(bottom=0)
axes[-1].set_xlabel('15-min timestep')
fig.suptitle('Sample smart meter series (first 2 weeks)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Average daily profile (all meters)

In [ ]:
STEPS_PER_DAY = 96
n_days = T // STEPS_PER_DAY
values = df.values[:n_days * STEPS_PER_DAY, :]   # (n_days*96, N)
daily = values.reshape(n_days, STEPS_PER_DAY, N) # (days, 96, N)

mean_profile = np.nanmean(daily, axis=(0, 2))     # (96,)
std_profile  = np.nanstd( daily, axis=(0, 2))

hours = np.arange(STEPS_PER_DAY) / 4             # 0..24
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(hours, mean_profile - std_profile,
                        mean_profile + std_profile, alpha=0.3)
ax.plot(hours, mean_profile, linewidth=2)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Raw consumption')
ax.set_title('Average daily consumption profile ± 1 std (all meters, all days)')
ax.set_xticks(range(0, 25, 2))
plt.tight_layout()
plt.show()

## 5. Seasonal heatmap (daily mean per meter)

In [ ]:
daily_mean = np.nanmean(daily, axis=1)  # (n_days, N)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(daily_mean.T, aspect='auto', cmap='viridis')
plt.colorbar(im, ax=ax, label='Daily mean consumption')
ax.set_xlabel('Day index')
ax.set_ylabel('Meter index')
ax.set_title('Seasonal heatmap: daily mean per meter')
plt.tight_layout()
plt.show()

## 6. Distribution summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Per-meter mean
axes[0].hist(np.nanmean(df.values, axis=0), bins=30, color='steelblue', edgecolor='w')
axes[0].set_xlabel('Per-meter mean consumption')
axes[0].set_title('Distribution of meter means')

# Per-meter std
axes[1].hist(np.nanstd(df.values, axis=0), bins=30, color='coral', edgecolor='w')
axes[1].set_xlabel('Per-meter std')
axes[1].set_title('Distribution of meter stds')

# Overall value distribution (subsample)
flat = df.values.ravel()
flat = flat[~np.isnan(flat)]
axes[2].hist(flat[::100], bins=50, color='mediumseagreen', edgecolor='w')
axes[2].set_xlabel('Consumption value')
axes[2].set_title('Overall value distribution')

plt.tight_layout()
plt.show()

print('Global min:', np.nanmin(df.values))
print('Global max:', np.nanmax(df.values))
print('Global mean:', np.nanmean(df.values))

## 7. Notes

> Fill in observations here after running the notebook.